# GWAS Workflow with MRBIGR2

End-to-end genome-wide association study: genotype QC, phenotype preparation, GWAS, visualization, and QTL detection.

**Prerequisites:** `pip install -e .` from the MRBIGR2 repo root.

In [ ]:
import os
import pandas as pd
from mrbigr.core import geno, pheno, gwas, vis, qtl, anno

# Adjust paths to your data
GENO_PREFIX = "data/chr_HAMP"
PHENO_FILE  = "data/pheno.csv"
GTF_FILE    = "data/annotation.gtf"  # optional
OUT_DIR     = "output/gwas_demo"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Genotype QC

In [ ]:
qc_prefix = f"{OUT_DIR}/qc"
geno.snp_qc(GENO_PREFIX, qc_prefix, maf=0.05, missing_rate=0.2, mind=0.2)
print("QC complete. Output:", qc_prefix)

## 2. PCA for population structure

In [ ]:
pc_df, var_ratio = geno.calculate_pca(qc_prefix, n_components=5)
print(f"Variance explained: {var_ratio[:5]}")
pc_df.head()

## 3. Phenotype preparation

In [ ]:
phe = pd.read_csv(PHENO_FILE)
phe = pheno.missing_filter(phe, 0.3)
phe = pheno.outlier(phe, method="zscore")
phe = pheno.scale_wrapper(phe, method="zscore")
print(f"Phenotype shape after prep: {phe.shape}")
phe.head()

## 4. Kinship matrix

In [ ]:
kin_prefix = f"{OUT_DIR}/kinship"
geno.calculate_kinship(qc_prefix, kin_prefix)

## 5. GWAS (Linear Mixed Model)

In [ ]:
result_files = gwas.gwas_lmm(phe, qc_prefix, output_dir=OUT_DIR)
print("GWAS result files:", result_files)

## 6. Visualization

In [ ]:
for f in (result_files or []):
    stem = os.path.splitext(f)[0]
    vis.manhattan_plot(f, output_file=f"{stem}_manhattan.png")
    vis.qq_plot(f, output_file=f"{stem}_qq.png")
    print(f"Plots saved for {f}")

## 7. QTL detection

In [ ]:
if result_files:
    qtl_df = qtl.detect_qtl(result_files[0], p1=1e-7, p2=1e-5)
    if qtl_df is not None:
        print(f"Detected {len(qtl_df)} QTL regions")
        display(qtl_df)
    else:
        print("No significant QTL detected at current thresholds.")

## 8. Gene annotation (optional)

In [ ]:
if qtl_df is not None and os.path.exists(GTF_FILE):
    genes = qtl.map_qtl_to_genes(qtl_df, GTF_FILE)
    display(genes)
else:
    print("Skipping annotation (no QTL or no GTF file).")